In [1]:
import pandas as pd
import numpy as np 

df=pd.read_csv("../data/processed/labeled_prs_v2.csv")
print(df.shape)

#yesterday's code check
print("First-timers:", round(df["is_first_pr"].mean(), 2))

(500, 18)
First-timers: 0.45


In [2]:
#embed PR titles
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df["title"].fillna("").tolist(), show_progress_bar=True)
print("Embeddings shape:", embeddings.shape) 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embeddings shape: (500, 384)


In [3]:
from sklearn.decomposition import PCA

numeric = ["additions", "deletions", "changed_files", "commits",
           "comments", "review_comments", "num_files",
           "author_past_prs", "author_past_bug_rate", "is_first_pr"]

y = df["label"].values

# version A: numeric + full 384-dim embeddings
X_full = np.hstack([df[numeric].values, embeddings])

# version B: numeric + embeddings compressed to 30 dims
emb_small = PCA(n_components=30, random_state=42).fit_transform(embeddings)
X_pca = np.hstack([df[numeric].values, emb_small])

print("Full:", X_full.shape, "| PCA:", X_pca.shape)

Full: (500, 394) | PCA: (500, 40)


In [4]:
#training
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

def evaluate(X, y, name):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    for model_name, m in [
        ("Logistic Regression", LogisticRegression(max_iter=3000)),
        ("Random Forest", RandomForestClassifier(n_estimators=200, random_state=42)),
    ]:
        m.fit(X_train, y_train)
        auc = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
        print(f"\n=== {model_name} — {name} ===")
        print(classification_report(y_test, m.predict(X_test)))
        print("AUC:", round(auc, 3))

In [5]:
evaluate(X_full, y, "numeric + full embeddings (394 features)")
evaluate(X_pca, y, "numeric + PCA embeddings (40 features)")


=== Logistic Regression — numeric + full embeddings (394 features) ===
              precision    recall  f1-score   support

           0       0.68      0.88      0.77        60
           1       0.68      0.38      0.48        40

    accuracy                           0.68       100
   macro avg       0.68      0.63      0.63       100
weighted avg       0.68      0.68      0.65       100

AUC: 0.659

=== Random Forest — numeric + full embeddings (394 features) ===
              precision    recall  f1-score   support

           0       0.64      0.83      0.72        60
           1       0.55      0.30      0.39        40

    accuracy                           0.62       100
   macro avg       0.59      0.57      0.56       100
weighted avg       0.60      0.62      0.59       100

AUC: 0.649

=== Logistic Regression — numeric + PCA embeddings (40 features) ===
              precision    recall  f1-score   support

           0       0.69      0.90      0.78        60
       